In [ ]:
# Install required packages
#!pip install -q -U qiskit qiskit-ibm-runtime
#!pip install -q matplotlib numpy ipython
#!pip install -q samplomatic
#!pip install -q qiskit-addon-utils qiskit-addon-pna qiskit-addon-slc
#!pip install --upgrade qc-grader
#!pip install plotly
#!pip install pylatexenc
#!pip install nbformat

In [ ]:
### Scientific Computing
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

### IPython
from IPython.display import display

### Qiskit 
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

### Qiskit IBM Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, Executor, QuantumProgram
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.options_models.noise_learner_v3_options import NoiseLearnerV3Options
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

### Samplomatic
import samplomatic
from samplomatic import Twirl, InjectNoise, ChangeBasis, build
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions, get_annotation

### Qiskit Addons
from qiskit_addon_utils.exp_vals.measurement_bases import get_measurement_bases
from qiskit_addon_utils.exp_vals.expectation_values import executor_expectation_values
from qiskit_addon_utils.noise_management import trex_factors, gamma_from_noisy_boxes
from qiskit_addon_pna import generate_noise_mitigating_observable
from qiskit_addon_slc.bounds import compute_backward_bounds, compute_forward_bounds, compute_local_scales, merge_bounds
from qiskit_addon_slc.utils import generate_noise_model_paulis, map_modifier_ref_to_ref
from qiskit_addon_slc.visualization import draw_shaded_lightcone


# Grader imports
from qc_grader.challenges.qgss_2026 import (
    grade_lab3_ex1,
    grade_lab3_ex2,
    grade_lab3_ex3,
    grade_lab3_ex4,
    grade_lab3_ex5,
)

# grader function to check your progress
from qc_grader.challenges.qgss_2026 import check_progress

#### Define the backend:

In this lab we will work with a __Heron device__. As stated above, noise learning jobs executed on the backend cannot be executed with a simulator. By focusing on a Heron device, participants working with an Open Plan account can participate in this lab. 

It is a good idea to you use the _same backend_ throughout this tutorial. We will be learning the noise in the backend and using that noise model to implement advanced error mitigation techniques. Therefore, if you change backend halfway through, the learned noise model will not be applicable to the new backend.

Note: If you are having trouble connecting to a backend, return to Lab 0 and make sure you have set up your IBM Quantum Platform Account correctly.

In [ ]:
service = QiskitRuntimeService()

# Define the backend: 
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    filters=lambda b: b.processor_type.get('family') == 'Heron'
)

print(f"Backend     : {backend.name}")
print(f"# qubits    : {backend.num_qubits}")
print(f"Basis gates : {backend.basis_gates}")

# Create ISA pass manager for transpilation with optimization_level=0
isa_pm = generate_preset_pass_manager(backend=backend, optimization_level=0)


In [ ]:
def dress(U: QuantumCircuit, v_in: str, v_out: str) -> QuantumCircuit:
    """Build the circuit  V_out . U . V_in  as a QuantumCircuit.
    """
    n = U.num_qubits
    assert not v_in.startswith(("+", "-", "i")), (
        f"v_in must be an unsigned Pauli string, got {v_in!r}"
    )
    assert not v_out.startswith(("+", "-", "i")), (
        f"v_out must be an unsigned Pauli string, got {v_out!r}"
    )
    assert len(v_in) == len(v_out) == n, "Pauli string length must match U's num_qubits"
    qc = QuantumCircuit(n)
    # Qiskit string 'q_{n-1}...q_0' — reverse to iterate in qubit order.
    for q, p in enumerate(reversed(v_in)):
        if p != "I":
            getattr(qc, p.lower())(q)
    qc.compose(U, inplace=True)
    for q, p in enumerate(reversed(v_out)):
        if p != "I":
            getattr(qc, p.lower())(q)
    return qc


def is_invariant(U: QuantumCircuit, v_in: str, v_out: str) -> bool:
    """True iff  V_out . U . V_in  equals  U  up to a global phase.

    Takes *unsigned* Pauli strings v_in, v_out. The invariance condition is
    U V_in U† = ±V_out; the ± becomes a global phase on the dressed operator,
    so it does not affect invariance. We use `Pauli.equiv`, which is
    phase-insensitive, to compare.
    """
    return Pauli(v_in).evolve(U, frame='s').equiv(Pauli(v_out))

## Chapter 2 — Samplomatic and NoiseLearner

To mitigate noise at the level of individual layers or gates, three things are needed: a way to address parts of a circuit separately, a measurement of the noise on each part, and a way to run the mitigated program and collect results. Samplomatic, `NoiseLearnerV3`, and the `Executor` primitive supply these three pieces. Together they form Qiskit Runtime's [directed execution model](https://quantum.cloud.ibm.com/docs/en/guides/directed-execution-model), which captures mitigation intent on the client side and shifts the generation of circuit variants to the server.

[Samplomatic](https://github.com/Qiskit/samplomatic) addresses parts of a circuit through **boxes** and **annotations**. A box is a region of a circuit marked off as a unit — a single gate, a layer, or the final measurement. An annotation is an instruction attached to a box: `Twirl` randomizes the box's dressing, `InjectNoise` declares that a learned noise channel will be applied here, `ChangeBasis` rotates the measurement. The instruction is attached to the box rather than the whole circuit, so each box can be treated separately. A `build` step then turns an annotated circuit into a parametric *template* plus a *samplex* — a recipe for filling the box parameters at run time.

[`NoiseLearnerV3`](https://quantum.cloud.ibm.com/docs/en/guides/noise-learning) supplies the measurement. It runs characterization circuits on the backend and returns a [Pauli-Lindblad noise model](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/utils-noise-learner-result-pauli-lindblad-error) for each box, addressed by the same boxes Samplomatic uses. The two fit together through one pattern: a box *declares* with `InjectNoise` that it expects a noise model, and the specific model measured by `NoiseLearnerV3` is plugged in at run time, matched by the `InjectNoise` reference.

The [`Executor`](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/executor) primitive runs the mitigated program itself. Both it and `NoiseLearnerV3` submit jobs to a backend, but for different purposes: `NoiseLearnerV3` runs circuits to *measure noise*, while the `Executor` runs the box-aware program built from a template and samplex to *produce the computation's result*. It is the [box-aware counterpart](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-runtime-primitives) to `Sampler` and `Estimator`. __Boxing__ is the structure shared across all three steps (how per-box mitigation is specified, how per-box noise is reported, and what the Executor runs) and it is what the Chapter 3 add-ons, propagated noise absorption (PNA) and shaded lightcones (SLC), act on to mitigate one layer or one observable at a time.

In the following sections, we will introduce each tool step by step for a small 2-qubit toy model. Then, in section 2.6, we will put them together on a 1D Ising chain, the system Chapter 3 continues with.

## 2.1  Boxes and the `Twirl` annotation

We begin with boxing up a circuit. Lets start with a very simple example: a 2-qubit circuit with a single CZ gate. We use the `.box` method to create a box around the CZ gate and tell it that it is annotated with a `Twirl` object.

In section 1.1.2, we introduced Pauli Twirling as an important tool for error mitigation. Pauli twirling a quantum circuit tailors the noise in the circuit to a _stochastic Pauli channel_. This is done by replacing a single circuit with a random ensemble of circuit, a _randomization_. Importantly, twirling alone is not expected the mitigate errors in the circuit but instead, tailor the noise to be mitigated effectively by other techniques. 

The `Twirl`annotation is used to indicate that the box should be _twirled_.

In [ ]:
toy = QuantumCircuit(2)
with toy.box(annotations=[Twirl()]):
    toy.cz(0, 1)

toy.draw("mpl")

The circuit diagram shows the box (red square) wrapping the CZ gate, but the `Twirl` annotation itself does not appear in the picture. It is metadata attached to the box. We can read it back by iterating over the circuit's instructions:

In [ ]:
for idx, instruction in enumerate(toy):
    print(f"Box {idx}: annotations = {instruction.operation.annotations}")

The instructions printed above tell us the following:

There is one box in this circuit, carrying a single annotation: `Twirl(group='pauli', dressing='left', decomposition='rzsx')`. These are the default `Twirl` settings. Lets break them down:
- `group='pauli'`: applies Pauli-group twirling 
- `dressing='left'`: the random dressing is on the left side of the box 
- `decomposition='rzsx'`: records that the dressing compiles down to the hardware's native `rz` and `sx` gates. 

The box marks *which* part of the circuit to twirl; the annotation records *how*. Turning that intent into runnable circuits is the next step.

### 2.1.1  Templates

The next step in Samplomatic workflow is to *build* a template circuit and a samplex. The `build(boxed_circuit)` method turns an annotated circuit into two objects: a *template* (a parametric `QuantumCircuit` with enough free gates to realize any single randomization) and a *samplex* (a recipe describing how to fill those parameters at run time). 

A common error when trying to use `build` is to have a dressing that doesn't have a place to go. We illustrate this in the below cell.

In [ ]:
try:
    template_circuit, samplex = build(toy)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

We see that `build` fails. The `Twirl` annotation instructs Samplomatic to place its random-Pauli dressing on the _left_ (input) side of the box. But, for the box's logical action to stay unchanged, that Pauli has to propagate through the CZ and come out the _right_ (output) side as a *virtual gate* that some later box must absorb. Here the circuit just ends after the box, so those right-going virtual gates have _nowhere to land_, hence we obtain a  `SamplexBuildError`. The error reports unterminated virtual gates on qubits `[0, 1]`. A `Twirl` box always needs a *collector*: another box that receives them. In the next cell we add one as a measurement box. We can then successfully implement `build`.

In [ ]:
toy = QuantumCircuit(2)
with toy.box(annotations=[Twirl()]):
    toy.cz(0, 1)
with toy.box(annotations=[Twirl()]):
    toy.measure_all()

template_circuit, samplex = build(toy)
template_circuit.draw("mpl", fold=-1)

Wrapping the measurement in its own `Twirl` box gives the right-side dressings a place to land: the virtual gates coming out of the CZ box are absorbed into the measurement box, which twirls the readout at the same time. `build` now succeeds, returning the `template` and its `samplex`. 

In the circuit diagram for the template, the single-qubit gates are decomposed into `rz` and `sx` and left as free parameters; the `samplex` holds the recipe that fills those parameters with a concrete random-Pauli assignment each time a randomization is drawn. One template plus one samplex stands in for the entire ensemble of randomized circuits.

### 2.1.2  The samplex


As we saw above, `build` returns two objects: the _template_ and the _samplex_. The samplex is the runtime recipe: a [directed acyclic graph](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.dagcircuit.DAGCircuit) (DAG) of inputs and outputs that determines, for each randomization, the parameter values to feed into the template circuit and the post-processing to apply to the measured bitstrings. 

In [ ]:
print(samplex)
fig = samplex.draw()
fig.show(renderer="notebook")
print(type(fig))

The printout text above shows the samplex's inputs and outputs and the DAG diagram shows them as a graph. The **Inputs** list is empty. This is because a samplex with only `Twirl` annotations needs no runtime data, unlike when `InjectNoise` annotations are added (which we will see in section 2.4). The two **Outputs**, `parameter_values` and `measurement_flips.meas`, are the collector nodes (bowties) in the diagram.

The DAG diagram is **not** a quantum circuit. It is the samplex's classical dataflow graph: the template (drawn above in section 2.1.1) is the circuit, and the samplex computes what fills its parameter slots for each randomization. So, a node in the samplex diagram is a computation step and an edge carries a register of classical data between steps.

Three kinds of nodes appear in the samplex:
- **Red stars** are *sampling* nodes. There is one per `Twirl` box, drawing that box's random Paulis. Our toy example has two boxes: the CZ box and the measurement box.
- **Green circles** are processing steps: propagating a Pauli past the CZ (the Clifford conjugation from 1.2, now visualized as a graph node), and slicing or combining registers. 
- **Bowties** are *collectors* that gather data into the outputs. The **blue** ones into `parameter_values` (`[num_randomizations, 12]`, the template's free parameters) and the **purple** one into `measurement_flips.meas` (`[num_randomizations, 1, 2]`, the bit-flips that undo the measurement twirling). 

Next is the sampling step. To do this, we use the method `samplex.sample(...)`, specifying the number of randomizations `num_randomizations`. Each call to `samplex.sample(...)` draws `num_randomizations` independent sets of random Paulis from the twirl distribution. It returns them already converted into the form the circuit needs: `parameter_values` to fill the template, and `measurement_flips` to correct the readout afterward. 

In [ ]:
outputs = samplex.sample({}, num_randomizations=5)

print("parameter_values.shape     :", outputs["parameter_values"].shape)
print("measurement_flips.meas.shape:", outputs["measurement_flips.meas"].shape)
print()
print("First two parameter draws:")
print(outputs["parameter_values"][:2])
print()
print("Bit-flip corrections:")
print(outputs["measurement_flips.meas"][:, 0, :])

Here, we sampled 5 randomizations. The output is a dictionary with two keys: `parameter_values` and `measurement_flips`. The template of the toy model has 12 parameters and the measurement twirl has 2 bits, so the output shapes are as expected. We have 5 sets of 12 parameters for the template, and 5 sets of 2 bit-flip corrections for the measurement twirl.

This is a purely classical step. No circuit has run yet. The template stays fixed; only the parameter values and measurement flips change from one randomization to the next. Running the filled-in circuits on hardware is the `Executor`'s job, in section 2.5.

Two questions follow naturally: how many randomizations to draw, and how to box a circuit without writing the `with circuit.box(...)` blocks by hand. The next section answers both.

## 2.2  Randomizations and the boxing pass manager

The first of those questions is how many randomizations to draw. Two parameters control this and are easy to confuse: `num_randomizations` (independent random circuits drawn from the twirl distribution) and `shots_per_randomization` (shots taken within each one). They trade off against each other. Past a certain ratio, the variance from twirling dominates the per-shot statistical noise, so adding randomizations helps more than adding shots.

The second question is how to box a circuit without writing `with circuit.box(...)` blocks by hand. We can do this with a [_boxing pass manager_](https://qiskit.github.io/samplomatic/guides/transpiler.html). `generate_boxing_pass_manager(...)` is a transpiler pass that boxes and annotates a circuit automatically. This section uses three of its options — `enable_gates`, `enable_measures`, and `twirling_strategy`. The `inject_noise_*` options are covered in section 2.3 and measurement annotations in section 2.4.

Let's box the same toy circuit as in section 2.1 — a CZ followed by a measurement — but this time automatically, with the pass manager instead of hand-written box blocks.

In [ ]:
# create the raw toy circuit
raw_toy = QuantumCircuit(2, 2)
raw_toy.cz(0, 1)
raw_toy.measure([0, 1], [0, 1])

# transpile the toy circuit using isa_pm we defined at the start of the lab
raw_toy_isa = isa_pm.run(raw_toy)

# create the boxing pass manager
twirl_only_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    twirling_strategy="active",
)

# box the toy circuit using the boxing pass manager
boxed_toy = twirl_only_pm.run(raw_toy_isa)
boxed_toy.draw("mpl", idle_wires=False)

The boxing pass manager takes the bare circuit (no boxes) and produces the same boxed circuit we built by hand in section 2.1, automatically. 

- `enable_gates=True` wraps the 2-qubit gate in a `Twirl` box
- `enable_measures=True` wraps the measurement in a box
- `twirling_strategy="active"` twirls only the qubits each box acts on

The circuit diagram shows the two resulting boxes in red, each of which have the default `Twirl` annotation (unseen in the circuit diagram). 

For a circuit with many layers using the boxing pass manager is the practical way to box. Writing `with circuit.box(...)` for every layer by hand does not scale well.